In [ ]:
import torch
import numpy as np
import pandas as pd
from typing import List, Tuple, Dict, Optional

In [ ]:
letters = "abcdefghijklmnopqrstuvwxyz"
padding = "P"
letters = letters+ padding

In [ ]:
from torch.utils.data import DataLoader, Dataset
from torch import nn

In [ ]:
words_file = "words_250000_train.txt"
words = []
with open(words_file, "r") as f:
    words = f.readlines()
words = list(map( lambda x: x.strip(), words))
# words = words[:1000]
# words

In [ ]:
gpt_config = {
    "vocab_size" : len(letters),
    "dataloader_batch_size" : 50,
    "context_size" : 8,
    "emb_dim":24,
    "context_dim":24,
    "num_heads":4,
    "qkv_bias" : False,
    "dropout_rate":0.1,
    "adamw_learning_rate":4e-4,
    "adamw_weight_decay":1e-1
}

In [ ]:
str_to_int = {l:i for i, l in enumerate(letters)}
int_to_str = {l:i for i, l in str_to_int.items()}
text_to_token = lambda w : [str_to_int[c] for c in w]
token_to_text = lambda idx : "".join([int_to_str[x] for x in idx])

### Step 1: Create tokenizer Dataset

In [ ]:
class words_dataset(Dataset):
    def __init__(self, words_dataset: List[str], cfg : Dict):
        super().__init__()
        self.input_batch = []
        self.target_batch = []

        for word in words:
            tokens = [str_to_int[c] for c in word]
            for i in range(2, len(tokens)-1): # start with 2+1 letter words, don't map the end-of-word to anything
                l_padding = max(cfg["context_size"] - i, 0)
                padding_array = [str_to_int[padding]] * l_padding
                input_chunk = padding_array + tokens[:i]
                input_chunk = input_chunk[-cfg["context_size"]:]
                target_chunk = input_chunk[-cfg["context_size"] + 1:] + [tokens[i]]
                self.input_batch.append(torch.tensor(input_chunk))
                self.target_batch.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_batch)

    def __getitem__(self, idx : int):
        return self.input_batch[idx], self.target_batch[idx]

### Step 2: Create DataLoader

In [ ]:
def create_dataloader(
    words : List[str],
    cfg : Dict,
    shuffle : bool= True,
    num_workers : int = 0,
    drop_last : bool = False
):
    dataset = words_dataset(words, cfg)
    return DataLoader(
        dataset,
        batch_size=cfg["dataloader_batch_size"],
        shuffle = shuffle,
        num_workers=num_workers,
        drop_last=drop_last
    )

train_load = create_dataloader(words[:-50_000], cfg=gpt_config)
val_load = create_dataloader(words[-50_000:], cfg = gpt_config)

### Step 3: Prepare Elements of the GPT model

#### 3.1 Multihead attention module

In [ ]:
class MultiheadAttentionModule(nn.Module):
    def __init__(
        self, embedding_dim, context_dim, num_heads, dropout_rate=0.05, qkv_bias=False
    ):
        super().__init__()
        self.w_key = nn.Linear(embedding_dim, context_dim, bias=qkv_bias)
        self.w_value = nn.Linear(embedding_dim, context_dim, bias=qkv_bias)
        self.w_query = nn.Linear(embedding_dim, context_dim, bias=qkv_bias)
        self.dropout = nn.Dropout(dropout_rate)
        self.embedding_dim = embedding_dim
        self.context_dim = context_dim
        self.num_heads = num_heads
        self.head_dim = self.context_dim // self.num_heads
        self.out_proj = nn.Linear(context_dim, context_dim)

    def forward(self, embedding_tensor):
        b, num_tokens, _ = embedding_tensor.shape  # we already know dim of embedding
        query = self.w_query(embedding_tensor)
        key = self.w_key(embedding_tensor)
        value = self.w_value(embedding_tensor)

        key = key.view(b, num_tokens, self.num_heads, self.head_dim)
        value = value.view(b, num_tokens, self.num_heads, self.head_dim)
        query = query.view(b, num_tokens, self.num_heads, self.head_dim)

        key = key.transpose(1, 2)
        value = value.transpose(1, 2)
        query = query.transpose(1, 2)

        # masked self-attention
        attn_score = query @ key.transpose(2, 3)

        # buffer, manually move to gpu...
        mask = torch.triu(
            torch.ones(num_tokens, num_tokens, device=embedding_tensor.device),
            diagonal=1,
        )
        attn_score.masked_fill(mask.bool(), -torch.inf)

        attn_weight = torch.softmax(attn_score / key.shape[-1] ** 0.5, dim=-1)
        attn_weight = self.dropout(attn_weight)

        context_vec = (attn_weight @ value).transpose(1, 2)
        context_vec = context_vec.contiguous().view(b, num_tokens, self.context_dim)
        context_vec = self.out_proj(context_vec)

        return context_vec

#### 3.2 GPT architecture elements

In [ ]:
# ====================================


class LayerNorm(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.eps = 1e-5
        self.scale = nn.Parameter(torch.ones(input_dim))
        self.shift = nn.Parameter(torch.zeros(input_dim))

    def forward(self, input_tensor):
        mean = input_tensor.mean(dim=-1, keepdim=True)
        var = input_tensor.var(dim=-1, keepdim=True, unbiased=False)
        layernorm_output = (input_tensor - mean) / torch.sqrt(var + self.eps)
        return self.scale * layernorm_output + self.shift


# ====================================


class GELU(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, input_tensor):
        return (
            0.5
            * input_tensor
            * (
                1
                + torch.tanh(
                    torch.sqrt(torch.tensor(2 / torch.pi))
                    * (input_tensor + 0.044715 * torch.pow(input_tensor, 3))
                )
            )
        )


# ====================================


class FeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.feed_forward = nn.Sequential(
            nn.Linear(cfg["emb_dim"], 2 * cfg["emb_dim"], bias=cfg["qkv_bias"]),
            GELU(),
            nn.Linear(2 * cfg["emb_dim"], cfg["emb_dim"], bias=cfg["qkv_bias"]),
        )

    def forward(self, input_tensor):
        return self.feed_forward(input_tensor)

#### 3.3 GPT Model

In [ ]:
class transformer_block(nn.Module):
    def __init__(self, cfg: Dict):
        super().__init__()
        self.ln1 = LayerNorm(cfg["emb_dim"])
        self.att1 = MultiheadAttentionModule(embedding_dim=cfg["emb_dim"],
                                    context_dim=cfg["context_dim"],
                                    num_heads=cfg["num_heads"],
                                    dropout_rate=cfg["dropout_rate"],
                                    qkv_bias=cfg["qkv_bias"]
        )
        self.ff1 = FeedForward(cfg)

    def forward(self, x):
        x = self.ln1(x)
        x = self.att1(x)
        x = self.ff1(x)
        return x

In [ ]:
class gpt_model(nn.Module):

    def __init__(self, cfg : Dict):

        super().__init__()
        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])
        self.pos_emb = nn.Embedding(cfg["context_size"], cfg["emb_dim"])

        self.trf_block1 = transformer_block(cfg)
        self.trf_block2 = transformer_block(cfg)
        self.trf_block3 = transformer_block(cfg)

        self.final_norm = LayerNorm(cfg["emb_dim"])
        self.out_head = nn.Linear(cfg["emb_dim"], cfg["vocab_size"], bias=False)

    def forward(self, input_tensor):

        token_embeds = self.tok_emb(input_tensor)
        pos_embeds = self.pos_emb(
            torch.arange(input_tensor.shape[1], device=input_tensor.device)
        )
        total_embeds = token_embeds + pos_embeds
        x = self.trf_block1(total_embeds)
        x = x + total_embeds  # skip connections so that the training is stable
        x = self.trf_block2(total_embeds)
        x = x + total_embeds
        x = self.trf_block3(total_embeds)

        x = self.final_norm(x)
        logits = self.out_head(x)

        return logits

### Step 4: Defining Loss functions

In [ ]:
def calc_loss_batch(
    input_batch: torch.Tensor, target_batch: torch.Tensor, model: gpt_model
):
    "Define the cross-entropy loss between the logits from inputs and target batch"

    input_batch = input_batch.long()
    target_batch = target_batch.long()

    logits = model(input_batch)

    loss = torch.nn.functional.cross_entropy(
        logits.flatten(0, 1), target_batch.flatten()
    )

    return loss


# ======================================


def calc_loss_loader(
    data_loader: torch.utils.data.DataLoader,
    model: gpt_model,
    device="cpu",
    num_batches: Optional[int] = None,
):
    "calculate loss function for a given set of batches together"

    total_loss = 0.0
    model = model.to(device)

    if len(data_loader) == 0:
        return float("nan")
    elif num_batches is None:
        num_batches = len(data_loader)
    else:
        num_batches = min(len(data_loader), num_batches)

    for i, (input_batch, target_batch) in enumerate(data_loader):
        # send data to device
        input_batch = input_batch.to(device)
        target_batch = target_batch.to(device)

        if i < num_batches:
            loss = calc_loss_batch(
                input_batch=input_batch, target_batch=target_batch, model=model
            )
            total_loss += loss.item()

        else:
            break

    return total_loss / num_batches

### Step 5 : Generation

In [ ]:
def generate_next_char(
    model: gpt_model,
    idx: torch.Tensor,
    max_new_tokens: int = 1,
    context_size: int = 8,
    temperature: float = 0.0,
    topk: Optional[int] = None,
    eos_id=None,
    result_with_input:bool = False,
) -> torch.Tensor:
    "generates next stock with top k sampling and temperature scaling"

    model.eval()

    for _ in range(max_new_tokens):
        idx_cond = idx[:, -context_size:]

        with torch.no_grad():
            logits = model(idx_cond)

        logits = logits[:, -1, :-1] # remove padding

        # print(logits.shape)
        if topk is not None:
            top_logits, _ = torch.topk(logits, topk)
            if top_logits.dim() == 1:
                top_logits = top_logits.unsqueeze(dim=0)
            min_val = top_logits[:, -1]
            min_val = torch.reshape(min_val, (min_val.shape[0], 1))  # for comparison
            logits = torch.where(
                logits < min_val[None, :], torch.tensor(float("-inf")), logits
            )

        if temperature > 0.0:
            logits = logits / temperature
            probs = torch.softmax(logits, dim=-1).squeeze(dim=0)
            # print(probs.shape)
            idx_next = torch.multinomial(probs, num_samples=1)
        else:
            idx_next = torch.argmax(logits, dim=-1, keepdim=True)

        if idx_next == eos_id:
            break

        idx = torch.cat([idx, idx_next], dim=1)

    return idx if result_with_input else idx[:, -max_new_tokens:]

### Finally, training

In [ ]:
class hangman_trainer:

    def __init__(
        self,
        model: gpt_model,
        cfg: dict,
        device="cpu"
    ):
        self.model = model
        self.device = device

        self.model = self.model.to(device)

        self.optimizer = torch.optim.AdamW(
            self.model.parameters(),
            lr=cfg["adamw_learning_rate"],
            weight_decay=cfg["adamw_weight_decay"],
        )

        self.tokens_seen = 0
        self.steps_trained = 0
        self.epochs = 0

        self.evaluated_train_loss = []
        self.evaluated_val_loss = []

    # -----------------------------------------------------

    def engage_training(
        self,
        train_loader: torch.utils.data.DataLoader,
        num_epochs: int = 1,
        val_loader=None,
        eval_freq=None,
        eval_iter=None,
    ):
        for epoch in range(num_epochs):
            # train setting
            self.model.train()

            for i, (input_batch, target_batch) in enumerate(train_loader):

                # send input batch and target batch to device
                input_batch = input_batch.to(self.device)
                target_batch = target_batch.to(self.device)

                self.optimizer.zero_grad()
                loss = calc_loss_batch(
                    input_batch=input_batch, target_batch=target_batch, model=self.model
                )

                loss.backward()  # calculate the gradients backpropagation
                self.optimizer.step()
                self.tokens_seen += input_batch.numel()
                self.steps_trained += 1

                # evaluate model
                if (
                    eval_freq is not None
                    and eval_iter is not None
                    and val_loader is not None
                    and self.steps_trained % eval_freq == 0
                ):
                    print(f"\nEpoch {epoch + 1} Step {self.steps_trained }>>> ")
                    self.evaluate_model(train_loader, val_loader, eval_iter)

        # finally
        self.epochs += num_epochs

    # --------------------------------------------

    def evaluate_model(self, train_loader, val_loader, eval_iter):
        self.model.eval()  # pause model training
        with torch.no_grad():
            train_loss = calc_loss_loader(
                data_loader=train_loader,
                model=self.model,
                num_batches=eval_iter,
                device=self.device,
            )
            val_loss = calc_loss_loader(
                data_loader=val_loader,
                model=self.model,
                num_batches=eval_iter,
                device=self.device,
            )

        self.evaluated_train_loss.append(train_loss)
        self.evaluated_val_loss.append(val_loss)

        print(f"Train Loss : {train_loss:.3f}")
        print(f"Val Loss : {val_loss:.3f}")

        self.model.train()

    # --------------------------------------------

    def __str__(self):
        return f"""
        ============================================================
            Training Statistics for this exercise >>>

            Device : {self.device}
            Total Epochs trained : {self.epochs}
            Total Steps trained : {self.steps_trained}
            Total number of tokens seen so far : {self.tokens_seen}
            Training Loss : {self.evaluated_train_loss}
            Validation Loss : {self.evaluated_val_loss}
        ============================================================

        """

    # --------------------------------------------

In [ ]:
forward_gpt  = gpt_model(gpt_config)
# LOAD FROM COMPUTER
# torch.load


# TRAINING IS DONE
# trainer = hangman_trainer(
#     model=forward_gpt,
#     cfg=gpt_config,
#     device="cuda" #training on gpu
# )

# trainer.engage_training(train_loader=train_load, val_loader= val_load, eval_freq=1000, eval_iter=10)
# print(trainer)


Epoch 1 Step 1000>>> 
Train Loss : 0.418
Val Loss : 0.424

Epoch 1 Step 2000>>> 
Train Loss : 0.367
Val Loss : 0.363

Epoch 1 Step 3000>>> 
Train Loss : 0.341
Val Loss : 0.345

Epoch 1 Step 4000>>> 
Train Loss : 0.341
Val Loss : 0.332

Epoch 1 Step 5000>>> 
Train Loss : 0.334
Val Loss : 0.337

Epoch 1 Step 6000>>> 
Train Loss : 0.324
Val Loss : 0.329

Epoch 1 Step 7000>>> 
Train Loss : 0.333
Val Loss : 0.329

Epoch 1 Step 8000>>> 
Train Loss : 0.324
Val Loss : 0.317

Epoch 1 Step 9000>>> 
Train Loss : 0.325
Val Loss : 0.317

Epoch 1 Step 10000>>> 
Train Loss : 0.305
Val Loss : 0.312

Epoch 1 Step 11000>>> 
Train Loss : 0.315
Val Loss : 0.308

Epoch 1 Step 12000>>> 
Train Loss : 0.310
Val Loss : 0.319

Epoch 1 Step 13000>>> 
Train Loss : 0.317
Val Loss : 0.305

Epoch 1 Step 14000>>> 
Train Loss : 0.313
Val Loss : 0.308

Epoch 1 Step 15000>>> 
Train Loss : 0.326
Val Loss : 0.314

Epoch 1 Step 16000>>> 
Train Loss : 0.315
Val Loss : 0.311

Epoch 1 Step 17000>>> 
Train Loss : 0.305
Val Lo

In [ ]:
torch.save(forward_gpt.state_dict(), "3_layer_forward_model_params.pth")

In [ ]:
#test it
w = "borewel"
next_token = generate_next_char(
    forward_gpt,
    idx=torch.tensor(text_to_token(w)).unsqueeze(dim= 0).to("cuda"),
    temperature=1.5,
    topk=5
)
next_character = int_to_str[ int(next_token.squeeze()) ]
next_character

'l'

# Train Reverse GPT

In [ ]:
reverse_words = list(map(reversed, words))
reverse_train_load = create_dataloader(reverse_words[:-50_000], cfg=gpt_config)
reverse_val_load = create_dataloader(reverse_words[-50_000:], cfg = gpt_config)

In [ ]:
reverse_gpt = gpt_model(gpt_config)
# torch.load()

# TRAINING DONE
# reverse_trainer = hangman_trainer(model = reverse_gpt,
#                                   cfg = gpt_config,
#                                   device = "cuda")
# reverse_trainer.engage_training(train_loader =reverse_train_load, val_loader = reverse_val_load, eval_iter = 1000, eval_freq = 50)
# print(reverse_trainer)


            Training Statistics for this exercise >>>

            Device : cuda
            Total Epochs trained : 1
            Total Steps trained : 28863
            Total number of tokens seen so far : 11545152
            Training Loss : []
            Validation Loss : []

        


In [ ]:
torch.save(reverse_gpt.state_dict(), "3_layer_reverse_model_params.pth")

# Make BERT-like model by combining both forward and reverse

In [ ]:
class bert_model(nn.Module):
    """combine forward and reverse model outputs and generate a new output.
       Input models must be trained already.
       Input tensors must be single dimensional, cannot use batch here. """

    def __init__(self, forward_gpt, reverse_gpt, gpt_config):
        super().__init__()
        self.forward_gpt = forward_gpt
        self.reverse_gpt = reverse_gpt
        self.gpt_config = gpt_config
    def forward(self, forward_input_tensor, reverse_input_tensor):
        "predict the letter in between - fill in the blanks "
        forward_logits = forward_gpt(forward_input_tensor)
        reverse_logits = reverse_gpt(reverse_input_tensor)

        with torch.no_grad():
            total_logits = forward_logits + reverse_logits
            total_logits = torch.softmax(total_logits, dim = -1)  #sum up the probabilities and then take the softamx

        return total_logits

In [ ]:
def generate_fill_1_char(
    model: bert_model,
    forward_idx : torch.Tensor, # 1D
    backward_idx : torch.Tensor, # 1D
    context_size: int = 8,
    temperature: float = 0.0,
    topk: Optional[int] = None
) -> torch.Tensor:

    "generates 1 character fill in the blank with top k sampling and temperature scaling"

    f_idx_cond = forward_idx[-context_size:]
    b_idx_cond = backward_idx[-context_size:] # reverses inside here.
    logits = model(f_idx_cond.unsqueeze(dim = 0), b_idx_cond.unsqueeze(dim = 0) )

    logits = logits[:, -1, :-1] # remove padding

    # print(logits.shape)
    if topk is not None:
        top_logits, _ = torch.topk(logits, topk)
        if top_logits.dim() == 1:
            top_logits = top_logits.unsqueeze(dim=0)
        min_val = top_logits[:, -1]
        min_val = torch.reshape(min_val, (min_val.shape[0], 1))  # for comparison
        logits = torch.where(
            logits < min_val[None, :], torch.tensor(float("-inf")), logits
        )

    if temperature > 0.0:
        logits = logits / temperature
        probs = torch.softmax(logits, dim=-1).squeeze(dim=0)
        # print(probs.shape)
        idx_next = torch.multinomial(probs, num_samples=1)
    else:
        idx_next = torch.argmax(logits, dim=-1, keepdim=True)

    return idx_next

In [ ]:
w = ["borewel"] # borewell
forward = text_to_token(w[0])
backward = text_to_token(w[1])

# fill_token = generate_fill_1_char(model = bert_model,
#                                   forward_idx=torch.tensor(forward),
#                                   backward_idx=torch.tensor(backward[::-1]), # reverse and sendit
#                                   temperature=2,
#                                   topk=5)

NameError: name 'text_to_token' is not defined

# Fantastic. Done